# Lesson 21: Zava Retail Capstone (Part 2) — FastMCP Tools & Hybrid Vector Search

In this notebook, you will:
1. Build `FastMCP` tools (`get_database_schema`, `execute_sales_query`, `get_store_performance`).
2. Embed Zava Retail products with **Google AI Studio `gemini-embedding-001`** (with deterministic offline fallback).
3. Run Hybrid Semantic + SQL filtered product search.

In [ ]:
!pip install -q -U "mcp<2" google-genai numpy

In [ ]:
import os
import numpy as np
from google import genai
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ZavaRetailTools")

PRODUCTS = [
    {"sku": "S1", "store": "seattle", "name": "Aero Trail Runner", "desc": "Waterproof trail running shoes with high grip", "price": 129.0},
    {"sku": "S2", "store": "seattle", "name": "StormShell Jacket", "desc": "Breathable rain jacket for hiking", "price": 185.0},
    {"sku": "S3", "store": "redmond", "name": "Aero Trail Runner", "desc": "Waterproof trail running shoes", "price": 129.0},
]

def embed_text(text: str, client=None) -> np.ndarray:
    if client:
        res = client.models.embed_content(model="gemini-embedding-001", contents=text)
        return np.array(res.embeddings[0].values, dtype=np.float32)
    # Deterministic offline bag-of-keywords fallback
    kws = ["running", "shoes", "waterproof", "rain", "jacket", "hiking"]
    vec = np.array([1.0 if k in text.lower() else 0.1 for k in kws], dtype=np.float32)
    return vec / np.linalg.norm(vec)

api_key = os.environ.get("GEMINI_API_KEY")
gclient = genai.Client(api_key=api_key) if api_key else None

q_vec = embed_text("comfortable waterproof running shoes for rain", gclient)
scored = []
for p in PRODUCTS:
    if p["store"] == "seattle" and p["price"] <= 150.0:
        p_vec = embed_text(p["desc"], gclient)
        sim = float(np.dot(q_vec, p_vec) / (np.linalg.norm(q_vec) * np.linalg.norm(p_vec)))
        scored.append((sim, p["name"], p["price"]))
scored.sort(reverse=True)
print("Top Seattle hybrid results:", scored)

### Challenge Exercise
Add an in-memory SHA-256 `embedding_cache` dictionary to `embed_text()` so repeated queries reuse cached vectors without calling the Gemini API twice.